# Stop using Python to test your strategy

The Kaggriculture environment runs at about **1.2 episodes per second**. If you
want to compare two strategies properly — a few hundred seeds, both seats, a
spread of opponents — that is several thousand episodes, and you are waiting an
hour for every question you ask.

I got tired of that and rewrote the environment in C++. It runs the same
episode in **0.18–0.30 ms instead of 815 ms**, and it is *bit-exact*: identical
money and identical market inventory at every one of the 720 steps, not
approximately right.

| | per episode | episodes/sec | speedup |
|---|---|---|---|
| Python `kaggle_environments` | 815 ms | 1.23 | 1× |
| this port, single core | 0.18–0.30 ms | 3,300–5,600 | **~2,700–4,500×** |
| this port, 24 cores, full agent logic | — | ~35,000 | — |

That is the difference between testing an idea overnight and testing it while
you read the rest of this notebook.

**This notebook contains the whole thing.** Every cell below writes a real file,
then we compile it, generate ground truth from the actual Python environment,
and check the port against it — end to end, in front of you. No strategy code,
no secrets: just the environment, so you can go faster.

---

## First: pin the environment

**The `kaggle_environments` preinstalled in Kaggle notebooks is not the version
the competition runs.** I found this the hard way — the notebook image defaults
to `startingMoney = 2000`, while every real competition replay I have
downloaded says **3000**. Other differences
follow from that: the town draws different shops, so different goods leave the
market.

If you test in a notebook without pinning, you are tuning against a different
game than the one you are submitting to.

So the first cell pins the version this port was validated against. Everything
below then agrees with the competition rather than with the image.

In [1]:
!pip install -q kaggle-environments==1.32.6
!python -c "import kaggle_environments, importlib.metadata as m; print('kaggle-environments', m.version('kaggle-environments'))"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 69.0 MB/s eta 0:00:00
kaggle-environments 1.32.6


---

## What I am proposing

**Write your submission in Python, as the competition requires — but stop doing
your *testing* in Python.**

The environment is the bottleneck, not your agent. Every question worth asking
("is this actually better, or did I get lucky?") needs hundreds or thousands of
episodes, and at 1.2 episodes per second you simply stop asking them.

So I ported the environment to C++ and made it *bit-exact* against the real one.
The strategy I submit is still ordinary Python. The C++ port is where I decide
which strategy that should be.

> **To be clear: this is not a submission and cannot be submitted.** It is an
> offline testing tool. Your competition entry stays exactly as it is.

## What I have actually done

1. **Ported the environment to C++** — header-only, no dependencies, ~880 lines
   across two files (`pyrandom.hpp`, `sim.hpp`).
2. **Verified it is bit-exact, not approximately right.** Feed the C++ port the
   exact actions a real Python episode produced, and it reproduces the money and
   market inventory of **both players at every one of the 720 steps**. The
   notebook proves this live: 5 agent pairings × 3 seeds, 15/15 exact.
3. **Measured the speedup honestly** — 815 ms/episode down to 0.18 ms, comparing
   C++ on its heaviest workload against Python running do-nothing agents, which
   is the most favourable case for Python.
4. **Packaged it so you can check it yourself** rather than take my word for it.
   The validation cell runs the real Python environment and diffs against the
   port in front of you.

This is my port, not an official one. It matches on everything I have tested,
and the notebook hands you the tooling to test it on your own episodes. If you
find a divergence, the port is wrong — please say so.

## How to run this notebook

**On Kaggle:** click *Copy & Edit*, then *Run All*. It takes about a minute.
Everything is preinstalled; nothing needs downloading. You should see:

- six `Writing ...` lines as the source files are created
- `compiled`
- fifteen lines of `PASS ... 719 steps exact`
- a C++ throughput figure in the thousands of episodes/sec
- a Python throughput figure around 1.2 episodes/sec

**On your own machine:** run the notebook once to write the files out, or copy
them by hand — every code cell below is `%%writefile <filename>`, so the cell
titles *are* the filenames. Then:

    g++ -O3 -march=native -std=c++17 -o validate validate.cpp
    g++ -O3 -march=native -std=c++17 -o bench bench.cpp
    python export_trace.py --agents starter,random 1 2 3   # ground truth
    ./validate replay_*.txt                                # must all say PASS
    ./bench replay_starter_1.txt 5000                      # throughput

You need a C++17 compiler and `kaggle_environments`. Nothing else.

**To validate against your own agent** (recommended before you trust it for
anything), point the exporter at your submission file:

    python export_trace.py --agents /path/to/main.py,starter 1 2 3
    ./validate replay_*.txt

If that passes, the port reproduces *your* agent exactly and you can search
against it with confidence.

**To use it for real**, include `sim.hpp` and drive `Sim` yourself:

    #include "sim.hpp"
    kag::Config cfg; cfg.seed = 12345;
    kag::Sim sim(cfg);
    while (!sim.st.done) sim.step(my_action_p0, my_action_p1);
    double bank = sim.reward(0);

## What is in here

| section | what it gives you |
|---|---|
| Three things that make this hard | the bugs that cost me days — read this if you port it yourself |
| `pyrandom.hpp` | CPython-compatible Mersenne Twister |
| `sim.hpp` | the environment itself |
| `validate.cpp` | step-by-step exactness check against real episodes |
| `bench.cpp` | throughput benchmark |
| `export_trace.py` | dumps ground truth out of the real Python environment |
| Compile / validate / benchmark | the whole thing running end to end |
| Caveats | what is not modelled, and what to update if the config changes |

There is no strategy code here. This is the environment only — the part everyone
needs and nobody wants to write twice.

---

## Why bother, concretely

At 1.2 eps/sec, a 2,000-episode comparison takes **27 minutes**. At 3,300
eps/sec it takes **0.6 seconds**. This changes which questions are worth asking,
not just how long they take:

- *Is this change actually better, or did I get lucky on 20 seeds?* Run 4,000
  paired episodes and find out.
- *Does it hold up in both seats, against 100 different opponents?* That is
  20,000 episodes — under a minute.
- *Can I search over strategies?* Evolutionary search needs hundreds of
  thousands of episodes. In Python that is a week. In C++ it is a coffee.

The honest catch: a fast simulator that is subtly *wrong* is worse than a slow
one that is right, because it will confidently point you at bad strategies. So
most of the work here went into exactness, and the notebook proves it rather
than asserting it.

---

## Three things that make this harder than it looks

I hit all three. Each one produced a port that looked fine and diverged
thousands of steps later, long after the cause.

### 1. The RNG has to be CPython's, not C++'s

Weed spawns and shop unlocks consume random numbers. `std::mt19937` is the same
core generator, but that is not enough — you need:

- `init_by_array` seeding (Python seeds from a key array, not a single word)
- the **53-bit** double construction Python's `random()` uses (`a*67108864+b`
  over `2^53`), not `mt / 2^32`
- the **rejection sampling** in `_randbelow`, which draws `getrandbits(k)` in a
  loop until the value is in range — so it consumes a *variable* number of words

Get the last one wrong and your RNG stream desynchronises the first time an
unlucky draw happens, which might be day 9.

### 2. Python dicts iterate in insertion order, and the simulation depends on it

Per-farmer inventories are dicts, and `DROP` plus the end-of-day flush walk them
in order. When the **100-item shed cap binds**, that order decides which items
win the last slots and which get destroyed.

A natural C++ port stores inventory as `int[N_ITEMS]` and iterates in item
order. That is a different order, so it discards *different goods*, so banks
diverge — but only in episodes where the shed actually fills. `Farm` below
models the insertion order explicitly with `inv_keys` / `inv_nkeys` for exactly
this reason. This one cost me a day.

### 3. Market orders resolve by slot index, across both players

`process_market` walks order slot `i` and resolves **both players'** slot-`i`
orders against the same pre-commit inventory. So which slot an order sits in
changes the price it gets. Resolving player-by-player instead produces perfectly
plausible, wrong numbers.

---

## `pyrandom.hpp` — CPython's Mersenne Twister

Everything in point 1 above. This is a direct transliteration of CPython's
`_randommodule.c`, and it is worth reading if you are porting anything else that
has to match Python's `random`.

In [2]:
%%writefile pyrandom.hpp
// CPython-compatible Mersenne Twister.
//
// The environment seeds `random.Random((seed * 1_000_003) ^ day)` and calls
// .random() (weed spawns) and .choice() (shop unlocks). To reproduce episodes
// bit-for-bit we must match CPython's _randommodule.c exactly: init_by_array
// seeding from the integer's 32-bit little-endian words, the 53-bit
// random() construction, and getrandbits/_randbelow rejection sampling.
#pragma once
#include <cstdint>
#include <vector>

namespace kag {

class PyRandom {
public:
    explicit PyRandom(uint64_t key) { seed(key); }

    void seed(uint64_t key) {
        // CPython: abs(n) -> array of 32-bit words, little endian. A zero key
        // still yields a one-word array [0].
        uint32_t words[3];
        int n = 0;
        if (key == 0) {
            words[0] = 0;
            n = 1;
        } else {
            while (key) {
                words[n++] = static_cast<uint32_t>(key & 0xffffffffu);
                key >>= 32;
            }
        }
        init_by_array(words, n);
    }

    uint32_t genrand_uint32() {
        uint32_t y;
        if (index_ >= N) {
            for (int k = 0; k < N - M; ++k) {
                y = (mt_[k] & UPPER) | (mt_[k + 1] & LOWER);
                mt_[k] = mt_[k + M] ^ (y >> 1) ^ ((y & 1u) ? MATRIX_A : 0u);
            }
            for (int k = N - M; k < N - 1; ++k) {
                y = (mt_[k] & UPPER) | (mt_[k + 1] & LOWER);
                mt_[k] = mt_[k + (M - N)] ^ (y >> 1) ^ ((y & 1u) ? MATRIX_A : 0u);
            }
            y = (mt_[N - 1] & UPPER) | (mt_[0] & LOWER);
            mt_[N - 1] = mt_[M - 1] ^ (y >> 1) ^ ((y & 1u) ? MATRIX_A : 0u);
            index_ = 0;
        }
        y = mt_[index_++];
        y ^= (y >> 11);
        y ^= (y << 7) & 0x9d2c5680u;
        y ^= (y << 15) & 0xefc60000u;
        y ^= (y >> 18);
        return y;
    }

    // random.random(): 53 bits of precision from two 32-bit draws.
    double random() {
        uint32_t a = genrand_uint32() >> 5;
        uint32_t b = genrand_uint32() >> 6;
        return (a * 67108864.0 + b) * (1.0 / 9007199254740992.0);
    }

    // random.getrandbits(k) for 0 < k <= 32.
    uint32_t getrandbits(int k) { return genrand_uint32() >> (32 - k); }

    // Random._randbelow_with_getrandbits(n), n > 0.
    uint32_t randbelow(uint32_t n) {
        int k = 0;
        for (uint32_t v = n; v; v >>= 1) ++k;   // n.bit_length()
        uint32_t r = getrandbits(k);
        while (r >= n) r = getrandbits(k);
        return r;
    }

    // random.choice(seq) -> index
    uint32_t choice_index(uint32_t len) { return randbelow(len); }

private:
    static constexpr int N = 624;
    static constexpr int M = 397;
    static constexpr uint32_t MATRIX_A = 0x9908b0dfu;
    static constexpr uint32_t UPPER = 0x80000000u;
    static constexpr uint32_t LOWER = 0x7fffffffu;

    uint32_t mt_[N];
    int index_ = N;

    void init_genrand(uint32_t s) {
        mt_[0] = s;
        for (int i = 1; i < N; ++i)
            mt_[i] = 1812433253u * (mt_[i - 1] ^ (mt_[i - 1] >> 30)) + static_cast<uint32_t>(i);
        index_ = N;
    }

    void init_by_array(const uint32_t* key, int keylen) {
        init_genrand(19650218u);
        int i = 1, j = 0;
        int k = (N > keylen) ? N : keylen;
        for (; k; --k) {
            mt_[i] = (mt_[i] ^ ((mt_[i - 1] ^ (mt_[i - 1] >> 30)) * 1664525u)) + key[j] + static_cast<uint32_t>(j);
            ++i; ++j;
            if (i >= N) { mt_[0] = mt_[N - 1]; i = 1; }
            if (j >= keylen) j = 0;
        }
        for (k = N - 1; k; --k) {
            mt_[i] = (mt_[i] ^ ((mt_[i - 1] ^ (mt_[i - 1] >> 30)) * 1566083941u)) - static_cast<uint32_t>(i);
            ++i;
            if (i >= N) { mt_[0] = mt_[N - 1]; i = 1; }
        }
        mt_[0] = 0x80000000u;
        index_ = N;
    }
};

}  // namespace kag

Writing pyrandom.hpp


---

## `sim.hpp` — the environment

The whole simulation: board, crops, animals, market price curves, the town, the
day cycle. ~765 lines, header-only.

A few landmarks:

- `Config` — the competition defaults, baked in. **Change these if the
  competition config changes.**
- `market_price()` — the price curve, with the amplitude table built once
- `Farm` — note `inv_keys` / `inv_nkeys`, which is point 2 above
- `Sim::step()` — one turn: unit actions, then market, then the day boundary
- `process_market()` — point 3 above; note the slot-index loop over both players
- `end_of_day()` — growth, weeds, shop unlocks, and the shed flush

In [3]:
%%writefile sim.hpp
// Kaggriculture simulator — a faithful C++ port of kaggriculture.py.
//
// Design goals, in order: (1) bit-identical results to the Python interpreter,
// (2) zero heap allocation per step, (3) trivially copyable state so search can
// snapshot and fork episodes.
#pragma once
#include <cmath>
#include <cstdint>
#include <cstring>
#include <algorithm>
#include <array>
#include "pyrandom.hpp"

namespace kag {

// ---------------------------------------------------------------- item space
// PRODUCTS order matches kaggriculture.py exactly. The first five products are
// also the five crops, in CROPS order, so a crop id doubles as a product id.
enum Item : uint8_t {
    WHEAT = 0, CARROT, TOMATO, STRAWBERRY, MELON, EGG, MILK, WOOL, FERTILIZER,
    GOOSE, COW, SHEEP, N_ITEMS
};
constexpr int N_PRODUCTS = 9;
constexpr int N_CROPS = 5;
constexpr int N_ANIMALS = 3;
inline bool is_crop(uint8_t i) { return i < N_CROPS; }
inline bool is_product(uint8_t i) { return i < N_PRODUCTS; }
inline bool is_animal(uint8_t i) { return i >= GOOSE && i < N_ITEMS; }

// ---------------------------------------------------------------- unit ops
enum Op : uint8_t {
    OP_PASS = 0, OP_NORTH, OP_SOUTH, OP_EAST, OP_WEST,
    OP_PICKUP, OP_DROP, OP_PLACE,
    OP_PLANT, OP_WATER, OP_HARVEST, OP_FERTILIZE, OP_DIG,
    OP_BUILD_COOP, OP_BUILD_PASTURE,
    OP_FEED, OP_COLLECT_FERTILIZER, OP_CARE, OP_INVALID
};

enum MOp : uint8_t {
    M_NONE = 0, M_HIRE, M_BUY_LAND, M_BUY_SEED, M_BUY_PRODUCT, M_BUY_ANIMAL, M_SELL
};

// ---------------------------------------------------------------- static data
struct CropDef { int seed, first_yield_day, max_yield_day, interval, max_yield; bool ongoing; };
inline constexpr CropDef CROPS[N_CROPS] = {
    {  10, 2,  4, 0, 6, false },  // WHEAT
    {  20, 2,  3, 0, 4, false },  // CARROT
    {  50, 8,  8, 1, 4, true  },  // TOMATO
    { 100, 10, 10, 2, 4, true },  // STRAWBERRY
    {  80, 10, 12, 0, 6, false }, // MELON
};

enum Structure : uint8_t { ST_COOP = 0, ST_PASTURE = 1 };
struct AnimalDef { int cost; Structure structure; int first_yield_day, interval, max_held; Item product; };
inline constexpr AnimalDef ANIMALS[N_ANIMALS] = {
    { 300, ST_COOP,    4, 1, 4, EGG  },  // GOOSE
    { 400, ST_PASTURE, 8, 2, 6, MILK },  // COW
    { 500, ST_PASTURE, 6, 3, 6, WOOL },  // SHEEP
};

enum Shape : uint8_t { F_LINEAR, F_SQ, F_SQRT, F_LOG, F_LOG10 };
struct MarketDef { double base; int I0; double T; Shape below_f; double below_t; Shape above_f; double above_t; };
inline constexpr MarketDef MARKET[N_PRODUCTS] = {
    {  25, 10000, 400, F_SQRT,   0.80, F_LOG,    0.20 },  // WHEAT
    {  35, 10000, 450, F_LOG,    0.20, F_SQRT,   0.70 },  // CARROT
    {  60, 10000, 200, F_LINEAR, 0.40, F_SQRT,   0.60 },  // TOMATO
    { 120, 10000, 100, F_SQRT,   0.70, F_LINEAR, 1.60 },  // STRAWBERRY
    { 250, 10000, 300, F_LOG,    0.20, F_SQ,     3.60 },  // MELON
    {  50, 10000, 332, F_LINEAR, 0.40, F_LOG,    0.20 },  // EGG
    { 160, 10000, 122, F_SQRT,   0.60, F_LINEAR, 1.60 },  // MILK
    { 200, 10000, 105, F_LOG,    0.20, F_SQ,     3.20 },  // WOOL
    { 100, 10000, 200, F_LINEAR, 0.40, F_LINEAR, 0.40 },  // FERTILIZER
};

inline double shape(Shape f, double x) {
    if (x < 0.0) x = 0.0;
    switch (f) {
        case F_LINEAR: return x;
        case F_SQ:     return x * x;
        case F_SQRT:   return std::sqrt(x);
        case F_LOG:    return std::log(1.0 + x);
        case F_LOG10:  return std::log10(1.0 + x);
    }
    return x;
}

// Amplitudes are derived constants; precompute once.
struct MarketAmp { double below, above; };
inline const std::array<MarketAmp, N_PRODUCTS>& market_amps() {
    static const std::array<MarketAmp, N_PRODUCTS> a = [] {
        std::array<MarketAmp, N_PRODUCTS> r{};
        for (int i = 0; i < N_PRODUCTS; ++i) {
            r[i].below = MARKET[i].below_t * MARKET[i].base / shape(MARKET[i].below_f, MARKET[i].T);
            r[i].above = MARKET[i].above_t * MARKET[i].base / shape(MARKET[i].above_f, MARKET[i].T);
        }
        return r;
    }();
    return a;
}

// Python: max(PRICE_FLOOR, int(round(price))). Python's round() on a float is
// round-half-to-even, which is exactly nearbyint's default mode.
inline int market_price(int item, int inv) {
    const MarketDef& p = MARKET[item];
    const MarketAmp& a = market_amps()[item];
    double price;
    if (inv < p.I0) price = p.base + a.below * shape(p.below_f, static_cast<double>(p.I0 - inv));
    else            price = p.base - a.above * shape(p.above_f, static_cast<double>(inv - p.I0));
    int v = static_cast<int>(std::nearbyint(price));
    return v < 1 ? 1 : v;
}

// ---------------------------------------------------------------- shops / town
enum ShopId : uint8_t {
    SHOP_BAKERY = 0, SHOP_BRUNCH_SPOT, SHOP_FARMERS_MARKET, SHOP_ICE_CREAM_SHOP,
    SHOP_PET_CAFE, SHOP_PIZZA_SHOP, SHOP_SMOOTHIE_SHOP, SHOP_YARN_STORE, N_SHOPS
};
// NOTE: this array is in sorted(SHOPS) order, because the environment unlocks
// with `rng.choice(sorted(SHOPS))`.
inline constexpr uint16_t SHOP_MASK[N_SHOPS] = {
    (1u << EGG) | (1u << WHEAT),                                        // BAKERY
    (1u << EGG) | (1u << WHEAT) | (1u << STRAWBERRY),                   // BRUNCH_SPOT
    (1u << WHEAT) | (1u << CARROT) | (1u << TOMATO) | (1u << STRAWBERRY),// FARMERS_MARKET
    (1u << STRAWBERRY) | (1u << MILK) | (1u << WHEAT),                  // ICE_CREAM_SHOP
    (1u << CARROT),                                                     // PET_CAFE
    (1u << MILK) | (1u << TOMATO) | (1u << WHEAT),                      // PIZZA_SHOP
    (1u << STRAWBERRY) | (1u << MILK),                                  // SMOOTHIE_SHOP
    (1u << WOOL),                                                       // YARN_STORE
};
inline constexpr int SHOP_MULT[N_SHOPS] = { 1, 1, 1, 1, 2, 1, 1, 2 };  // single-product shops pull 2x
constexpr int MAX_SHOP_INSTANCES = 8;

// ---------------------------------------------------------------- config
struct Config {
    int episode_steps = 720;
    int board_size = 10;
    int starting_money = 3000;
    int max_orders = 10;
    int turns_per_day = 24;
    int shed_capacity = 100;
    double weed_chance = 0.005;
    int shop_unlock_interval = 3;
    int shop_sell_interval = 4;
    int center_sell_interval = 24;
    int hire_mult = 1;
    uint64_t seed = 0;
};

// ---------------------------------------------------------------- state
enum TileKind : uint8_t { T_EMPTY = 0, T_LOCKED, T_WEED, T_COOP, T_PASTURE, T_PLANT };

struct Tile {
    TileKind kind = T_EMPTY;
    uint8_t  what = 0;          // crop id (PLANT) or animal id (COOP/PASTURE with animal)
    bool     has_animal = false;
    bool     watered_today = false;
    bool     fed_today = false;
    bool     cared_today = false;
    bool     fertilizer_available = false;
    int8_t   consecutive_dry = 0;    // unwatered (plant) / unfed (animal)
    int8_t   yield_units = 0;
    int8_t   pending_care_bonus = 0;
    int16_t  planted_day = 0;        // or placed_day
    int32_t  max_lifespan_step = -1;
    int16_t  fertilized_until_day = -1;
};

constexpr int MAX_UNITS = 40;   // farmer + hands; hires are Fibonacci-priced so this is ample
constexpr int BOARD = 10;

struct Farm {
    double money = 0;
    Tile tiles[BOARD][BOARD];
    int8_t pos_x[MAX_UNITS], pos_y[MAX_UNITS];
    int n_units = 1;                 // index 0 is the main farmer
    int n_quadrants = 1;             // NW always unlocked
    int hires_today = 0;

    int16_t shed[N_ITEMS] = {0};
    int shed_total = 0;
    int16_t seeds[N_CROPS] = {0};
    int16_t inv[MAX_UNITS][N_ITEMS] = {{0}};
    // Per-unit inventories are Python dicts, and several code paths iterate
    // them in INSERTION order. That order decides which items win the last
    // slots when the 100-item shed cap binds, so it is load-bearing, not
    // cosmetic, and must be modelled explicitly.
    uint8_t inv_keys[MAX_UNITS][N_ITEMS] = {{0}};
    uint8_t inv_nkeys[MAX_UNITS] = {0};

    // Instrumentation only - never read by the interpreter, so behaviour and
    // parity are unaffected. `discarded` is the production the 100-item shed
    // cap silently destroyed, which is otherwise invisible to search.
    int32_t discarded[N_ITEMS] = {0};
    int32_t produced[N_ITEMS] = {0};
    int32_t sold_units[N_ITEMS] = {0};
    double  sell_revenue = 0;   // coins actually received from SELLs
    double  total_spend = 0;    // coins actually paid out

    void inv_add(int u, int item, int n) {
        if (n <= 0) return;
        if (inv[u][item] == 0) inv_keys[u][inv_nkeys[u]++] = (uint8_t)item;
        inv[u][item] += (int16_t)n;
    }
    void inv_erase(int u, int item) {
        inv[u][item] = 0;
        int k = 0;
        while (k < inv_nkeys[u] && inv_keys[u][k] != item) ++k;
        if (k == inv_nkeys[u]) return;
        for (int j = k; j + 1 < inv_nkeys[u]; ++j) inv_keys[u][j] = inv_keys[u][j + 1];
        inv_nkeys[u]--;
    }
    bool inv_take(int u, int item, int n) {
        if (inv[u][item] < n) return false;
        inv[u][item] -= (int16_t)n;
        if (inv[u][item] == 0) inv_erase(u, item);
        return true;
    }
    void inv_clear(int u) { for (int i = 0; i < N_ITEMS; ++i) inv[u][i] = 0; inv_nkeys[u] = 0; }
};

struct Market {
    int32_t inventory[N_PRODUCTS];
    int32_t prices[N_PRODUCTS];
};

struct State {
    Farm farms[2];
    Market market;
    uint8_t shops[MAX_SHOP_INSTANCES];
    int n_shops = 0;
    int step = 0;
    int day = 0;
    int hour = 0;
    bool done = false;
};

// ---------------------------------------------------------------- actions
struct UnitAction { uint8_t op = OP_PASS; uint8_t arg = 0; int16_t n = 1; };
struct Order { uint8_t op = M_NONE; uint8_t item = 0; int32_t n = 0; };

struct Action {
    UnitAction units[MAX_UNITS];     // units[0] = farmer
    int n_units = 1;
    Order orders[16];
    int n_orders = 0;
    void clear() { n_units = 1; units[0] = UnitAction{}; n_orders = 0; }
};

// ---------------------------------------------------------------- helpers
inline int quadrant_of(int x, int y, int bs) {
    // 0=NW 1=NE 2=SW 3=SE
    int half = bs / 2;
    return (y < half ? 0 : 2) + (x < half ? 0 : 1);
}
// Land is unlocked in the order NE, SW, SE.
inline constexpr int LAND_ORDER[3] = { 1, 2, 3 };
inline constexpr int LAND_PRICES[3] = { 1000, 2000, 4000 };

inline void shed_access_tiles(int bs, int out[4][2]) {
    int h = bs / 2;
    out[0][0] = h - 1; out[0][1] = h - 1;
    out[1][0] = h;     out[1][1] = h - 1;
    out[2][0] = h - 1; out[2][1] = h;
    out[3][0] = h;     out[3][1] = h;
}
inline bool is_shed_adjacent(int x, int y, int bs) {
    int h = bs / 2;
    return (x == h - 1 || x == h) && (y == h - 1 || y == h);
}

inline int fib(int n) { int a = 1, b = 1; for (int i = 0; i < n; ++i) { int t = b; b = a + b; a = t; } return a; }

class Sim {
public:
    Config cfg;
    State st;

    explicit Sim(const Config& c = Config{}) : cfg(c) { reset(); }

    void reset() {
        st = State{};
        int h = cfg.board_size / 2;
        for (int p = 0; p < 2; ++p) {
            Farm& f = st.farms[p];
            f.money = cfg.starting_money;
            for (int y = 0; y < cfg.board_size; ++y)
                for (int x = 0; x < cfg.board_size; ++x)
                    f.tiles[y][x].kind = (quadrant_of(x, y, cfg.board_size) == 0) ? T_EMPTY : T_LOCKED;
            f.n_units = 1;
            f.pos_x[0] = static_cast<int8_t>(h - 1);
            f.pos_y[0] = static_cast<int8_t>(h - 1);
        }
        for (int i = 0; i < N_PRODUCTS; ++i) {
            st.market.inventory[i] = MARKET[i].I0;
            st.market.prices[i] = static_cast<int>(MARKET[i].base);
        }
    }

    double reward(int p) const { return st.farms[p].money; }

    // One environment step, given both players' actions.
    void step(const Action& a0, const Action& a1) {
        if (st.done) return;
        const Action* acts[2] = { &a0, &a1 };
        int step_i = st.step;
        int day = step_i / cfg.turns_per_day;

        for (int p = 0; p < 2; ++p) apply_unit_actions(p, *acts[p], day);
        process_market(*acts[0], *acts[1]);
        town_consume(step_i);
        for (int p = 0; p < 2; ++p) decay_plants(st.farms[p], step_i);
        if ((step_i + 1) % cfg.turns_per_day == 0) end_of_day(day);

        st.step = step_i + 1;
        st.day = st.step / cfg.turns_per_day;
        st.hour = st.step % cfg.turns_per_day;
        if (step_i >= cfg.episode_steps - 2) st.done = true;
    }

private:
    // ------------------------------------------------------------ unit actions
    void apply_unit_actions(int p, const Action& a, int day) {
        Farm& f = st.farms[p];
        // Atomic PLANT validation: if requests for a crop exceed seeds held,
        // every PLANT of that crop this turn is dropped.
        // Demand is counted over every submitted unit action, including ones
        // addressed to hands that do not exist (matching the Python, where the
        // blocked set is computed before the position lookup no-ops).
        int demand[N_CROPS] = {0};
        for (int i = 0; i < a.n_units; ++i)
            if (a.units[i].op == OP_PLANT && a.units[i].arg < N_CROPS) demand[a.units[i].arg]++;
        int n = std::min(a.n_units, f.n_units);
        bool blocked[N_CROPS];
        for (int c = 0; c < N_CROPS; ++c) blocked[c] = demand[c] > f.seeds[c];

        for (int i = 0; i < n; ++i) {
            UnitAction u = a.units[i];
            if (u.op == OP_PLANT && u.arg < N_CROPS && blocked[u.arg]) continue;
            apply_unit(f, i, u, day);
        }
    }

    void apply_unit(Farm& f, int idx, const UnitAction& u, int day) {
        const int bs = cfg.board_size;
        int fx = f.pos_x[idx], fy = f.pos_y[idx];
        int16_t* inv = f.inv[idx];

        switch (u.op) {
            case OP_PASS: return;
            case OP_NORTH: case OP_SOUTH: case OP_EAST: case OP_WEST: {
                int nx = fx + (u.op == OP_EAST) - (u.op == OP_WEST);
                int ny = fy + (u.op == OP_SOUTH) - (u.op == OP_NORTH);
                // Movement onto LOCKED tiles is legal (hands can spawn there).
                if (nx < 0 || nx >= bs || ny < 0 || ny >= bs) return;
                f.pos_x[idx] = static_cast<int8_t>(nx);
                f.pos_y[idx] = static_cast<int8_t>(ny);
                return;
            }
            default: break;
        }

        Tile& tile = f.tiles[fy][fx];

        // Shed ops resolve before the LOCKED guard: three of the four
        // shed-access tiles start locked, and the shed itself is always owned.
        if (u.op == OP_DROP) {
            if (!is_shed_adjacent(fx, fy, bs)) return;
            // Insertion order, matching `for item, n in list(inv.items())`.
            uint8_t keys[N_ITEMS];
            int nk = f.inv_nkeys[idx];
            for (int k = 0; k < nk; ++k) keys[k] = f.inv_keys[idx][k];
            for (int k = 0; k < nk; ++k) {
                int it = keys[k];
                if (inv[it] <= 0) { f.inv_erase(idx, it); continue; }
                int room = std::max(0, cfg.shed_capacity - f.shed_total);
                int take = std::min<int>(inv[it], room);
                if (take > 0) { f.shed[it] += take; f.shed_total += take; }
                f.discarded[it] += inv[it] - take;
                f.inv_erase(idx, it);
            }
            return;
        }
        if (u.op == OP_PICKUP) {
            if (!is_shed_adjacent(fx, fy, bs)) return;
            if (u.n <= 0 || u.arg >= N_ITEMS) return;
            int take = std::min<int>(u.n, f.shed[u.arg]);
            if (take <= 0) return;
            f.shed[u.arg] -= take; f.shed_total -= take;
            f.inv_add(idx, u.arg, take);
            return;
        }
        if (u.op == OP_PLACE) {
            if (u.arg >= N_ITEMS) return;
            if (is_animal(u.arg)) {
                const AnimalDef& ad = ANIMALS[u.arg - GOOSE];
                TileKind want = (ad.structure == ST_COOP) ? T_COOP : T_PASTURE;
                if (tile.kind == want && !tile.has_animal) {
                    if (f.inv_take(idx, u.arg, 1)) {
                        Tile t{};
                        t.kind = want; t.what = u.arg; t.has_animal = true;
                        t.planted_day = static_cast<int16_t>(day);
                        tile = t;
                    }
                    return;
                }
            }
            if (is_shed_adjacent(fx, fy, bs)) {
                int n = std::min<int>(u.n, inv[u.arg]);
                if (n <= 0) return;
                n = std::min(n, std::max(0, cfg.shed_capacity - f.shed_total));
                if (n <= 0) return;
                f.inv_take(idx, u.arg, n);
                f.shed[u.arg] += n; f.shed_total += n;
            }
            return;
        }

        if (tile.kind == T_LOCKED) return;

        switch (u.op) {
            case OP_PLANT: {
                if (u.arg >= N_CROPS || tile.kind != T_EMPTY || f.seeds[u.arg] <= 0) return;
                f.seeds[u.arg] -= 1;
                const CropDef& cd = CROPS[u.arg];
                Tile t{};
                t.kind = T_PLANT; t.what = u.arg;
                t.planted_day = static_cast<int16_t>(day);
                t.consecutive_dry = 1;                 // planting day counts as unwatered
                t.yield_units = cd.ongoing ? 0 : 1;
                t.max_lifespan_step = cd.ongoing ? -1
                    : (day + cd.max_yield_day + 1) * cfg.turns_per_day;
                tile = t;
                return;
            }
            case OP_WATER: {
                if (tile.kind != T_PLANT || tile.watered_today) return;
                tile.watered_today = true;
                const CropDef& cd = CROPS[tile.what];
                if (!cd.ongoing) {
                    int age = day - tile.planted_day;
                    int w0 = (cd.max_yield_day + 1) / 2;
                    if (age >= w0 && age <= cd.max_yield_day) {
                        int bonus = (tile.fertilized_until_day >= day) ? 2 : 1;
                        tile.yield_units = static_cast<int8_t>(std::min(cd.max_yield, tile.yield_units + bonus));
                    }
                }
                return;
            }
            case OP_HARVEST: {
                if (tile.kind == T_EMPTY || tile.kind == T_WEED) return;
                if (tile.yield_units <= 0) return;
                if (tile.kind == T_PLANT) {
                    const CropDef& cd = CROPS[tile.what];
                    if (day - tile.planted_day < cd.first_yield_day) return;
                    f.inv_add(idx, tile.what, tile.yield_units);
                    f.produced[tile.what] += tile.yield_units;
                    tile.yield_units = 0;
                    if (!cd.ongoing) { Tile e{}; e.kind = T_EMPTY; tile = e; }
                } else if (tile.has_animal) {
                    f.inv_add(idx, ANIMALS[tile.what - GOOSE].product, tile.yield_units);
                    f.produced[ANIMALS[tile.what - GOOSE].product] += tile.yield_units;
                    tile.yield_units = 0;
                }
                return;
            }
            case OP_FERTILIZE: {
                if (tile.kind != T_PLANT) return;
                if (!f.inv_take(idx, FERTILIZER, 1)) return;
                tile.fertilized_until_day = std::max<int16_t>(tile.fertilized_until_day,
                                                             static_cast<int16_t>(day + 2));
                return;
            }
            case OP_DIG: {
                if (tile.kind == T_EMPTY) return;
                if (tile.has_animal) return;           // animals are not removable
                Tile e{}; e.kind = T_EMPTY; tile = e;
                return;
            }
            case OP_BUILD_COOP:
                if (tile.kind != T_EMPTY) return;
                { Tile t{}; t.kind = T_COOP; tile = t; }
                return;
            case OP_BUILD_PASTURE:
                if (tile.kind != T_EMPTY) return;
                { Tile t{}; t.kind = T_PASTURE; tile = t; }
                return;
            case OP_FEED:
                if (!tile.has_animal || tile.fed_today) return;
                if (!f.inv_take(idx, WHEAT, 1)) return;
                tile.fed_today = true;
                return;
            case OP_COLLECT_FERTILIZER:
                if (!tile.has_animal || !tile.fertilizer_available) return;
                tile.fertilizer_available = false;
                f.inv_add(idx, FERTILIZER, 1);
                f.produced[FERTILIZER] += 1;
                return;
            case OP_CARE:
                if (!tile.has_animal || tile.cared_today) return;
                tile.cared_today = true;
                return;
            default: return;
        }
    }

    // ------------------------------------------------------------ market
    struct OState { uint8_t type; uint8_t item; int32_t remaining; bool live; };

    void process_market(const Action& a0, const Action& a1) {
        const Action* acts[2] = { &a0, &a1 };
        int nq[2];
        for (int p = 0; p < 2; ++p) nq[p] = std::min(acts[p]->n_orders, cfg.max_orders);
        int max_len = std::max(nq[0], nq[1]);

        for (int i = 0; i < max_len; ++i) {
            OState os[2];
            for (int p = 0; p < 2; ++p) {
                os[p].live = false;
                if (i < nq[p]) {
                    const Order& o = acts[p]->orders[i];
                    if (o.op == M_HIRE || o.op == M_BUY_LAND) {
                        os[p].type = o.op; os[p].live = true; os[p].remaining = 1;
                    } else if (o.op != M_NONE && o.n > 0) {
                        os[p].type = o.op; os[p].item = o.item; os[p].remaining = o.n; os[p].live = true;
                    }
                }
            }
            // Atomic orders resolve once, in player order.
            for (int p = 0; p < 2; ++p) {
                if (!os[p].live) continue;
                if (os[p].type == M_HIRE) { do_hire(st.farms[p]); os[p].live = false; }
                else if (os[p].type == M_BUY_LAND) { do_buy_land(st.farms[p]); os[p].live = false; }
            }

            // Per-unit lockstep: both players see the same pre-commit inventory.
            for (;;) {
                struct Q { bool ok = false; uint8_t type = 0; uint8_t item = 0; int price = 0; };
                Q q[2];
                for (int p = 0; p < 2; ++p) {
                    if (!os[p].live || os[p].remaining <= 0) continue;
                    uint8_t t = os[p].type, it = os[p].item;
                    if (t == M_SELL && is_product(it)) {
                        q[p] = { true, t, it, market_price(it, st.market.inventory[it]) };
                    } else if (t == M_BUY_PRODUCT && (it == WHEAT || it == FERTILIZER)) {
                        // Quoted at post-buy inventory so a round trip nets zero.
                        q[p] = { true, t, it, market_price(it, st.market.inventory[it] - 1) };
                    } else if (t == M_BUY_SEED && is_crop(it)) {
                        q[p] = { true, t, it, CROPS[it].seed };
                    } else if (t == M_BUY_ANIMAL && is_animal(it)) {
                        q[p] = { true, t, it, ANIMALS[it - GOOSE].cost };
                    } else {
                        os[p].live = false;
                    }
                }
                if (!q[0].ok && !q[1].ok) break;
                bool committed = false;
                for (int p = 0; p < 2; ++p) {
                    if (!q[p].ok) continue;
                    if (commit_unit(q[p].type, q[p].item, q[p].price, st.farms[p])) {
                        os[p].remaining -= 1; committed = true;
                    } else {
                        os[p].live = false;
                    }
                }
                if (!committed) break;
            }
            refresh_prices();
        }
    }

    bool commit_unit(uint8_t op, uint8_t item, int price, Farm& f) {
        switch (op) {
            case M_SELL:
                if (f.shed[item] <= 0) return false;
                f.shed[item] -= 1; f.shed_total -= 1;
                f.money += price;
                f.sold_units[item] += 1;
                f.sell_revenue += price;
                if (price > 1) st.market.inventory[item] += 1;   // $1 sales don't add supply
                return true;
            case M_BUY_PRODUCT:
                if (f.money < price) return false;
                if (f.shed_total >= cfg.shed_capacity) return false;
                f.money -= price; f.total_spend += price;
                f.shed[item] += 1; f.shed_total += 1;
                st.market.inventory[item] -= 1;
                return true;
            case M_BUY_SEED:
                if (f.money < price) return false;
                f.money -= price; f.total_spend += price; f.seeds[item] += 1;
                return true;
            case M_BUY_ANIMAL:
                if (f.money < price) return false;
                if (f.shed_total >= cfg.shed_capacity) return false;
                f.money -= price; f.total_spend += price;
                f.shed[item] += 1; f.shed_total += 1;
                return true;
        }
        return false;
    }

    void refresh_prices() {
        for (int i = 0; i < N_PRODUCTS; ++i)
            st.market.prices[i] = market_price(i, st.market.inventory[i]);
    }

    void do_hire(Farm& f) {
        int cost = cfg.hire_mult * fib(f.hires_today);
        if (f.money < cost || f.n_units >= MAX_UNITS) return;
        f.money -= cost; f.total_spend += cost;
        f.hires_today += 1;
        // Spawn on the first free shed-access tile (NWSE), ties by occupancy.
        int acc[4][2]; shed_access_tiles(cfg.board_size, acc);
        int occ[4] = {0,0,0,0};
        for (int u = 0; u < f.n_units; ++u)
            for (int k = 0; k < 4; ++k)
                if (f.pos_x[u] == acc[k][0] && f.pos_y[u] == acc[k][1]) occ[k]++;
        int best = 0;
        for (int k = 1; k < 4; ++k) if (occ[k] < occ[best]) best = k;
        int idx = f.n_units++;
        f.pos_x[idx] = static_cast<int8_t>(acc[best][0]);
        f.pos_y[idx] = static_cast<int8_t>(acc[best][1]);
        f.inv_clear(idx);
    }

    void do_buy_land(Farm& f) {
        int extra = f.n_quadrants - 1;
        if (extra >= 3) return;
        int cost = LAND_PRICES[extra];
        if (f.money < cost) return;
        f.money -= cost; f.total_spend += cost;
        int quad = LAND_ORDER[extra];
        f.n_quadrants += 1;
        for (int y = 0; y < cfg.board_size; ++y)
            for (int x = 0; x < cfg.board_size; ++x)
                if (quadrant_of(x, y, cfg.board_size) == quad && f.tiles[y][x].kind == T_LOCKED)
                    f.tiles[y][x].kind = T_EMPTY;
    }

    // ------------------------------------------------------------ town / decay
    void town_consume(int step_i) {
        if (step_i % cfg.shop_sell_interval == 0) {
            for (int s = 0; s < st.n_shops; ++s) {
                uint16_t mask = SHOP_MASK[st.shops[s]];
                int mult = SHOP_MULT[st.shops[s]];
                for (int it = 0; it < N_PRODUCTS; ++it)
                    if (mask & (1u << it)) st.market.inventory[it] -= mult;
            }
        }
        if (step_i % cfg.center_sell_interval == 0) {
            for (int it = 0; it < N_PRODUCTS; ++it)
                if (it != FERTILIZER) st.market.inventory[it] -= 1;
        }
        refresh_prices();
    }

    void decay_plants(Farm& f, int step_i) {
        for (int y = 0; y < cfg.board_size; ++y)
            for (int x = 0; x < cfg.board_size; ++x) {
                Tile& t = f.tiles[y][x];
                if (t.kind != T_PLANT) continue;
                if (t.max_lifespan_step < 0 || step_i < t.max_lifespan_step) continue;
                if ((step_i - t.max_lifespan_step) % 2 != 0) continue;
                t.yield_units -= 1;
                if (t.yield_units <= 0) { Tile w{}; w.kind = T_WEED; t = w; }
            }
    }

    void end_of_day(int day) {
        PyRandom rng((cfg.seed * 1000003ull) ^ static_cast<uint64_t>(day));
        for (int p = 0; p < 2; ++p) {
            Farm& f = st.farms[p];
            daily_refresh_plants(f, day);
            daily_refresh_animals(f, day);
            spawn_weeds(f, rng);
            drop_inventories(f);
            int h = cfg.board_size / 2;
            f.n_units = 1;
            f.pos_x[0] = static_cast<int8_t>(h - 1);
            f.pos_y[0] = static_cast<int8_t>(h - 1);
            f.hires_today = 0;
            for (int u = 0; u < MAX_UNITS; ++u) f.inv_clear(u);
        }
        int next_day = day + 1;
        if (next_day > 0 && next_day % cfg.shop_unlock_interval == 0 && st.n_shops < MAX_SHOP_INSTANCES)
            st.shops[st.n_shops++] = static_cast<uint8_t>(rng.choice_index(N_SHOPS));
    }

    void daily_refresh_plants(Farm& f, int day) {
        int next_day = day + 1;
        for (int y = 0; y < cfg.board_size; ++y)
            for (int x = 0; x < cfg.board_size; ++x) {
                Tile& t = f.tiles[y][x];
                if (t.kind != T_PLANT) continue;
                bool was_watered = t.watered_today;
                t.consecutive_dry = was_watered ? 0 : static_cast<int8_t>(t.consecutive_dry + 1);
                t.watered_today = false;
                if (t.consecutive_dry >= 2) { Tile w{}; w.kind = T_WEED; t = w; continue; }
                const CropDef& cd = CROPS[t.what];
                if (!cd.ongoing) continue;
                int since = next_day - t.planted_day - cd.first_yield_day;
                if (since < 0) continue;
                if (since % cd.interval != 0) continue;
                int count = since / cd.interval + 1;
                if (count > cd.max_yield) continue;
                bool fert = was_watered && t.fertilized_until_day >= day;
                t.yield_units = static_cast<int8_t>(std::min(cd.max_yield, t.yield_units + (fert ? 2 : 1)));
                if (count == cd.max_yield) t.max_lifespan_step = (next_day + 1) * cfg.turns_per_day;
            }
    }

    void daily_refresh_animals(Farm& f, int day) {
        int next_day = day + 1;
        for (int y = 0; y < cfg.board_size; ++y)
            for (int x = 0; x < cfg.board_size; ++x) {
                Tile& t = f.tiles[y][x];
                if (!t.has_animal) continue;
                t.consecutive_dry = t.fed_today ? 0 : static_cast<int8_t>(t.consecutive_dry + 1);
                if (t.consecutive_dry >= 2) {           // escapes; structure remains
                    TileKind k = t.kind;
                    Tile s{}; s.kind = k; t = s;
                    continue;
                }
                const AnimalDef& ad = ANIMALS[t.what - GOOSE];
                int since = next_day - t.planted_day - ad.first_yield_day;
                if (since >= 0 && since % ad.interval == 0) {
                    int bonus = t.fed_today ? t.pending_care_bonus : 0;
                    t.yield_units = static_cast<int8_t>(std::min(ad.max_held, t.yield_units + 1 + bonus));
                    t.pending_care_bonus = 0;
                }
                if (t.cared_today && t.fed_today) t.pending_care_bonus += 1;
                t.fertilizer_available = true;
                t.fed_today = false;
                t.cared_today = false;
            }
    }

    void spawn_weeds(Farm& f, PyRandom& rng) {
        for (int y = 0; y < cfg.board_size; ++y)
            for (int x = 0; x < cfg.board_size; ++x)
                if (f.tiles[y][x].kind == T_EMPTY && rng.random() < cfg.weed_chance)
                    f.tiles[y][x].kind = T_WEED;
    }

    void drop_inventories(Farm& f) {
        // Insertion order again: with the shed near capacity this decides which
        // goods survive the day and which are discarded.
        for (int u = 0; u < f.n_units; ++u) {
            uint8_t keys[N_ITEMS];
            int nk = f.inv_nkeys[u];
            for (int k = 0; k < nk; ++k) keys[k] = f.inv_keys[u][k];
            for (int k = 0; k < nk; ++k) {
                int it = keys[k];
                if (f.inv[u][it] <= 0) { f.inv_erase(u, it); continue; }
                int room = std::max(0, cfg.shed_capacity - f.shed_total);
                int take = std::min<int>(f.inv[u][it], room);
                if (take > 0) { f.shed[it] += take; f.shed_total += take; }
                f.discarded[it] += f.inv[u][it] - take;
                f.inv_erase(u, it);
            }
        }
    }
};

}  // namespace kag

Writing sim.hpp


---

## `validate.cpp` — proof, not assertion

Replays a recorded episode's exact actions and compares money and market
inventory **at every step**, failing at the first divergence with the step
number and what disagreed. This is the file that matters: a fast simulator you
have not validated is a liability.

In [4]:
%%writefile validate.cpp
// Replay the exact actions a Python episode emitted and assert the C++ sim
// reproduces its money and market-inventory trajectory step for step.
#include "sim.hpp"
#include <cstdio>
#include <fstream>
#include <sstream>
#include <string>
#include <vector>

using namespace kag;

struct Turn { Action a[2]; };

// Older trace files have no CONFIG line. Peek for the token and rewind if it is
// absent, so both formats load.
static void read_config(std::istream& in, Config& cfg) {
    std::streampos pos = in.tellg();
    std::string tok;
    if (!(in >> tok)) return;
    if (tok != "CONFIG") { in.clear(); in.seekg(pos); return; }
    in >> cfg.episode_steps >> cfg.board_size >> cfg.starting_money
       >> cfg.max_orders >> cfg.turns_per_day >> cfg.shed_capacity
       >> cfg.weed_chance >> cfg.shop_unlock_interval
       >> cfg.shop_sell_interval >> cfg.center_sell_interval >> cfg.hire_mult;
}


static bool load(const std::string& path, uint64_t& seed, Config& cfg,
                 std::vector<Turn>& turns,
                 std::vector<std::array<double, 2>>& money,
                 std::vector<std::array<int, N_PRODUCTS>>& inv) {
    std::ifstream in(path);
    if (!in) { std::fprintf(stderr, "cannot open %s\n", path.c_str()); return false; }
    int n;
    in >> seed >> n;
    read_config(in, cfg);
    turns.resize(n);
    for (int t = 0; t < n; ++t)
        for (int p = 0; p < 2; ++p) {
            int nu, no; in >> nu >> no;
            Action& a = turns[t].a[p];
            a.n_units = std::min(nu, MAX_UNITS);
            for (int i = 0; i < nu; ++i) {
                int op, arg, cnt; in >> op >> arg >> cnt;
                if (i < MAX_UNITS) {
                    a.units[i].op = static_cast<uint8_t>(op);
                    a.units[i].arg = static_cast<uint8_t>(arg);
                    a.units[i].n = static_cast<int16_t>(cnt);
                }
            }
            a.n_orders = std::min(no, 16);
            for (int i = 0; i < no; ++i) {
                int op, item, cnt; in >> op >> item >> cnt;
                if (i < 16) {
                    a.orders[i].op = static_cast<uint8_t>(op);
                    a.orders[i].item = static_cast<uint8_t>(item);
                    a.orders[i].n = cnt;
                }
            }
        }
    std::string tag; in >> tag;                 // "TRUTH"
    money.clear(); inv.clear();
    double m0, m1;
    while (in >> m0 >> m1) {
        money.push_back({m0, m1});
        std::array<int, N_PRODUCTS> row{};
        for (int i = 0; i < N_PRODUCTS; ++i) in >> row[i];
        inv.push_back(row);
    }
    return true;
}

int main(int argc, char** argv) {
    int failures = 0;
    for (int f = 1; f < argc; ++f) {
        uint64_t seed = 0;
        std::vector<Turn> turns;
        std::vector<std::array<double, 2>> money;
        std::vector<std::array<int, N_PRODUCTS>> inv;
        Config cfg;                       // overwritten by the file's CONFIG line
        if (!load(argv[f], seed, cfg, turns, money, inv)) { ++failures; continue; }
        cfg.seed = seed;
        Sim sim(cfg);

        int first_bad = -1;
        const char* what = "";
        for (size_t t = 0; t < turns.size(); ++t) {
            // Compare BEFORE stepping: state at index t must match truth[t].
            for (int p = 0; p < 2 && first_bad < 0; ++p)
                if (sim.st.farms[p].money != money[t][p]) { first_bad = (int)t; what = "money"; }
            for (int i = 0; i < N_PRODUCTS && first_bad < 0; ++i)
                if (sim.st.market.inventory[i] != inv[t][i]) { first_bad = (int)t; what = "inventory"; }
            if (first_bad >= 0) break;
            sim.step(turns[t].a[0], turns[t].a[1]);
        }
        size_t last = money.size() - 1;
        bool final_ok = first_bad < 0 &&
                        sim.st.farms[0].money == money[last][0] &&
                        sim.st.farms[1].money == money[last][1];

        if (first_bad >= 0) {
            std::printf("FAIL %s  seed=%llu  first %s mismatch at step %d\n",
                        argv[f], (unsigned long long)seed, what, first_bad);
            std::printf("     cpp money=(%.0f, %.0f)  py money=(%.0f, %.0f)\n",
                        sim.st.farms[0].money, sim.st.farms[1].money,
                        money[first_bad][0], money[first_bad][1]);
            std::printf("     cpp inv=");
            for (int i = 0; i < N_PRODUCTS; ++i) std::printf("%d ", sim.st.market.inventory[i]);
            std::printf("\n     py  inv=");
            for (int i = 0; i < N_PRODUCTS; ++i) std::printf("%d ", inv[first_bad][i]);
            std::printf("\n");
            ++failures;
        } else if (!final_ok) {
            std::printf("FAIL %s  final money cpp=(%.0f, %.0f) py=(%.0f, %.0f)\n", argv[f],
                        sim.st.farms[0].money, sim.st.farms[1].money, money[last][0], money[last][1]);
            ++failures;
        } else {
            std::printf("PASS %s  seed=%-8llu  %zu steps exact   final=(%.0f, %.0f)\n",
                        argv[f], (unsigned long long)seed, turns.size(),
                        sim.st.farms[0].money, sim.st.farms[1].money);
        }
    }
    return failures ? 1 : 0;
}

Writing validate.cpp


---

## `bench.cpp` — throughput

In [5]:
%%writefile bench.cpp
// Throughput benchmark: replay a recorded episode as fast as possible.
#include "sim.hpp"
#include <chrono>
#include <cstdio>
#include <fstream>
#include <string>
#include <vector>

using namespace kag;
struct Turn { Action a[2]; };

// Older trace files have no CONFIG line. Peek for the token and rewind if it is
// absent, so both formats load.
static void read_config(std::istream& in, Config& cfg) {
    std::streampos pos = in.tellg();
    std::string tok;
    if (!(in >> tok)) return;
    if (tok != "CONFIG") { in.clear(); in.seekg(pos); return; }
    in >> cfg.episode_steps >> cfg.board_size >> cfg.starting_money
       >> cfg.max_orders >> cfg.turns_per_day >> cfg.shed_capacity
       >> cfg.weed_chance >> cfg.shop_unlock_interval
       >> cfg.shop_sell_interval >> cfg.center_sell_interval >> cfg.hire_mult;
}


static void load(const std::string& path, uint64_t& seed, Config& cfg,
                 std::vector<Turn>& turns) {
    std::ifstream in(path);
    int n; in >> seed >> n;
    read_config(in, cfg);
    turns.resize(n);
    for (int t = 0; t < n; ++t)
        for (int p = 0; p < 2; ++p) {
            int nu, no; in >> nu >> no;
            Action& a = turns[t].a[p];
            a.n_units = std::min(nu, MAX_UNITS);
            for (int i = 0; i < nu; ++i) {
                int op, arg, cnt; in >> op >> arg >> cnt;
                if (i < MAX_UNITS) { a.units[i] = { (uint8_t)op, (uint8_t)arg, (int16_t)cnt }; }
            }
            a.n_orders = std::min(no, 16);
            for (int i = 0; i < no; ++i) {
                int op, item, cnt; in >> op >> item >> cnt;
                if (i < 16) a.orders[i] = { (uint8_t)op, (uint8_t)item, cnt };
            }
        }
}

int main(int argc, char** argv) {
    uint64_t seed; std::vector<Turn> turns;
    Config base;                      // overwritten by the file's CONFIG line
    load(argc > 1 ? argv[1] : "../data/replay_70117.txt", seed, base, turns);
    int reps = argc > 2 ? std::atoi(argv[2]) : 2000;

    double sink = 0;
    auto t0 = std::chrono::steady_clock::now();
    for (int r = 0; r < reps; ++r) {
        Config cfg = base; cfg.seed = seed + r;   // vary seed so weeds actually re-roll
        Sim sim(cfg);
        for (auto& tn : turns) sim.step(tn.a[0], tn.a[1]);
        sink += sim.st.farms[0].money;
    }
    auto dt = std::chrono::duration<double>(std::chrono::steady_clock::now() - t0).count();
    std::printf("%d episodes in %.3fs  =  %.0f eps/sec  (%.3f ms/episode)   [sink %.0f]\n",
                reps, dt, reps / dt, dt / reps * 1000, sink);
    return 0;
}

Writing bench.cpp


---

## `export_trace.py` — ground truth from the real environment

Runs the actual Python environment and records what the built-in agents
*actually did*, plus the resulting trajectory. We then feed those exact actions
to the C++ port and require it to reproduce the trajectory.

Recording emitted actions rather than re-running an agent matters: the reference
agents are not pure replays (they repair around weeds and reorder SELL slots),
so re-running one need not produce the same action sequence twice.

In [6]:
%%writefile export_trace.py
"""Export ground truth from the real Python environment, for validating the port.

Runs a full episode with the built-in agents, records the actions they ACTUALLY
emitted, and dumps them alongside the resulting money and market-inventory
trajectory. Feeding those exact actions to the C++ simulator must reproduce the
trajectory step for step — that is what `validate` checks.

Recording emitted actions rather than replaying an agent matters: the reference
agents are not pure replays (they repair around weeds and reorder SELL slots),
so re-running one is not guaranteed to produce the same action sequence.

    python export_trace.py 70117 1 2 3        # writes replay_<agent>_<seed>.txt

Usage: export_trace.py [--agents A,B] seed [seed ...]
"""
import contextlib, io, sys
from pathlib import Path

# These orderings must match the enums in sim.hpp exactly.
OPS = ["PASS", "NORTH", "SOUTH", "EAST", "WEST", "PICKUP", "DROP", "PLACE",
       "PLANT", "WATER", "HARVEST", "FERTILIZE", "DIG",
       "BUILD_COOP", "BUILD_PASTURE", "FEED", "COLLECT_FERTILIZER", "CARE"]
ITEMS = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL",
         "FERTILIZER", "GOOSE", "COW", "SHEEP"]
MOPS = ["NONE", "HIRE", "BUY_LAND", "BUY_SEED", "BUY_PRODUCT", "BUY_ANIMAL", "SELL"]
OPI = {o: i for i, o in enumerate(OPS)}
ITI = {o: i for i, o in enumerate(ITEMS)}
MOPI = {o: i for i, o in enumerate(MOPS)}


def enc_unit(a):
    if not isinstance(a, list) or not a:
        return (0, 0, 1)
    op = OPI.get(a[0], len(OPS))
    arg = ITI.get(a[1], 255) if len(a) >= 2 and isinstance(a[1], str) else 0
    try:
        n = int(a[2]) if len(a) >= 3 else 1
    except (TypeError, ValueError):
        n = 1
    return (op, arg, n)


def enc_order(o):
    if not isinstance(o, list) or not o:
        return (0, 0, 0)
    op = MOPI.get(o[0], 0)
    if op in (1, 2):            # HIRE and BUY_LAND carry no item or quantity
        return (op, 0, 1)
    if len(o) < 3:
        return (0, 0, 0)
    return (op, ITI.get(o[1], 255), int(o[2]))


def export(seed, agents=("starter", "random"), out_dir="."):
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        from kaggle_environments import make
        env = make("kaggriculture", configuration={"episodeSteps": 720, "seed": seed},
                   debug=False)
        env.run(list(agents))

    n = len(env.steps) - 1                      # number of acting turns
    # Emit the environment's ACTUAL configuration rather than letting the C++
    # side assume defaults. Package versions disagree - a Kaggle notebook image
    # and a local install differed on startingMoney (2000 vs 3000) - and a
    # hardcoded constant turns that into a silent, total miscalibration.
    cfg = env.configuration
    conf = [cfg["episodeSteps"], cfg["boardSize"], cfg["startingMoney"],
            cfg["maxMarketOrdersPerTurn"], cfg["turnsPerDay"], cfg["shedCapacity"],
            cfg["weedSpawnChance"], cfg["townShopUnlockInterval"],
            cfg["townShopSellInterval"], cfg["townCenterSellInterval"],
            cfg["farmHandCostMult"]]
    lines = [f"{seed} {n}", "CONFIG " + " ".join(str(v) for v in conf)]
    for t in range(n):
        for seat in (0, 1):
            # The action that drives steps[t] -> steps[t+1] is recorded on the
            # RESULTING state, not the originating one.
            act = env.steps[t + 1][seat].action or {}
            units = [act.get("farmer") or ["PASS"]] + list(act.get("hands") or [])
            orders = list(act.get("market") or [])
            parts = [str(len(units)), str(len(orders))]
            for u in units:
                parts += [str(v) for v in enc_unit(u)]
            for o in orders:
                parts += [str(v) for v in enc_order(o)]
            lines.append(" ".join(parts))

    money = [[float(s[0].observation.farms[0]["money"]),
              float(s[0].observation.farms[1]["money"])] for s in env.steps]
    inv = [[int(v) for v in s[0].observation.market["inventory"].values()]
           for s in env.steps]
    lines.append("TRUTH")
    for i in range(len(money)):
        lines.append(" ".join([f"{money[i][0]:.0f}", f"{money[i][1]:.0f}"] +
                              [str(v) for v in inv[i]]))

    p = Path(out_dir) / f"replay_{agents[0]}_{seed}.txt"
    p.write_text("\n".join(lines) + "\n")
    print(f"seed {seed}: final money {money[-1]}  -> {p.name}")
    return p


def main():
    args = sys.argv[1:]
    agents = ("starter", "random")
    if args and args[0] == "--agents":
        args.pop(0)
        agents = tuple(args.pop(0).split(","))
    seeds = [int(x) for x in (args or [70117])]
    for s in seeds:
        export(s, agents)


if __name__ == "__main__":
    main()

Writing export_trace.py


---

## Compile it

Any C++17 compiler. No dependencies.

In [7]:
!g++ -O3 -march=native -std=c++17 -o validate validate.cpp
!g++ -O3 -march=native -std=c++17 -o bench bench.cpp
print("compiled")

compiled


---

## Generate ground truth from the real Python environment

Five agent pairings × three seeds. Nothing here is cherry-picked — change the
seeds and rerun if you doubt it.

In [8]:
import subprocess, itertools

pairs = [("starter","random"), ("random","starter"), ("starter","starter"),
         ("random","random"), ("pass","starter")]
seeds = [11, 22, 33]

for a, b in pairs:
    subprocess.run(["python", "export_trace.py", "--agents", f"{a},{b}"] +
                   [str(s) for s in seeds], check=True)
    for s in seeds:
        subprocess.run(["mv", f"replay_{a}_{s}.txt", f"rp_{a}_{b}_{s}.txt"], check=True)
print("ground truth written")

seed 11: final money [3470.0, 0.0]  -> replay_starter_11.txt
seed 22: final money [3455.0, 0.0]  -> replay_starter_22.txt
seed 33: final money [3510.0, 0.0]  -> replay_starter_33.txt
seed 11: final money [0.0, 3476.0]  -> replay_random_11.txt
seed 22: final money [0.0, 3506.0]  -> replay_random_22.txt
seed 33: final money [0.0, 3500.0]  -> replay_random_33.txt
seed 11: final money [3479.0, 3479.0]  -> replay_starter_11.txt
seed 22: final money [3502.0, 3502.0]  -> replay_starter_22.txt
seed 33: final money [3510.0, 3510.0]  -> replay_starter_33.txt
seed 11: final money [0.0, 20.0]  -> replay_random_11.txt
seed 22: final money [0.0, 0.0]  -> replay_random_22.txt
seed 33: final money [0.0, 0.0]  -> replay_random_33.txt
seed 11: final money [3000.0, 3511.0]  -> replay_pass_11.txt
seed 22: final money [3000.0, 3483.0]  -> replay_pass_22.txt
seed 33: final money [3000.0, 3512.0]  -> replay_pass_33.txt
ground truth written


---

## Validate: every step, every episode

`PASS ... 719 steps exact` means the C++ simulator produced identical money and
identical market inventory at all 720 states. Any divergence at any step is a
failure with the step number reported.

In [9]:
!./validate rp_*.txt

PASS rp_pass_starter_11.txt  seed=11        719 steps exact   final=(3000, 3511)
PASS rp_pass_starter_22.txt  seed=22        719 steps exact   final=(3000, 3483)
PASS rp_pass_starter_33.txt  seed=33        719 steps exact   final=(3000, 3512)
PASS rp_random_random_11.txt  seed=11        719 steps exact   final=(0, 20)
PASS rp_random_random_22.txt  seed=22        719 steps exact   final=(0, 0)
PASS rp_random_random_33.txt  seed=33        719 steps exact   final=(0, 0)
PASS rp_random_starter_11.txt  seed=11        719 steps exact   final=(0, 3476)
PASS rp_random_starter_22.txt  seed=22        719 steps exact   final=(0, 3506)
PASS rp_random_starter_33.txt  seed=33        719 steps exact   final=(0, 3500)
PASS rp_starter_random_11.txt  seed=11        719 steps exact   final=(3470, 0)
PASS rp_starter_random_22.txt  seed=22        719 steps exact   final=(3455, 0)
PASS rp_starter_random_33.txt  seed=33        719 steps exact   final=(3510, 0)
PASS rp_starter_starter_11.txt  seed=11        7

---

## Now the speed

Same episode, same work. First the port:

In [10]:
!./bench rp_starter_random_11.txt 5000

5000 episodes in 2.204s  =  2269 eps/sec  (0.441 ms/episode)   [sink 17437817]


And the Python environment, doing as little as possible — the agents below
literally return `PASS`, so this is the *most* favourable case for Python. Real
agents are slower.

In [11]:
%%writefile bench_python.py
import contextlib, io, time
from kaggle_environments import make

# The lightest possible agents, so this times the ENVIRONMENT and not a policy.
# That makes it the most favourable case for Python; real agents are slower.
def noop(obs, config=None):
    return {"farmer": ["PASS"], "hands": [], "market": []}

with contextlib.redirect_stdout(io.StringIO()):        # warm up
    make("kaggriculture", configuration={"episodeSteps": 720, "seed": 1},
         debug=False).run([noop, noop])

N = 12
t0 = time.perf_counter()
for i in range(N):
    with contextlib.redirect_stdout(io.StringIO()):
        env = make("kaggriculture",
                   configuration={"episodeSteps": 720, "seed": i}, debug=False)
        env.run([noop, noop])
dt = time.perf_counter() - t0
print(f"{N} episodes in {dt:.2f}s  =  {N/dt:.2f} eps/sec  ({1000*dt/N:.0f} ms/episode)")

Writing bench_python.py


In [12]:
!python bench_python.py

12 episodes in 28.15s  =  0.43 eps/sec  (2346 ms/episode)


---

## What this bought me

Some things I could only find because episodes were cheap:

- **Opponent replays are worth more than intuition.** Extracting the top
  player's trajectories and simulating against them told me exactly where I
  stood — 0 wins in 1,400 games, at one point — which no amount of self-play
  against my own agent would have revealed.
- **Most in-house improvements are noise.** With thousands of paired episodes
  you can actually run a significance test, and most "improvements" do not
  survive one. That saved me from several bad submissions. It did not save me
  from all of them.
- **Validation sets go stale and lie.** A set of opponents you already beat will
  happily certify a change that does nothing. Cheap episodes let you re-validate
  against the *current* field instead of the one you built a week ago.

None of that needs C++ specifically. It needs episodes to be cheap enough that
you ask questions properly instead of guessing.

---

## Caveats — please read

- **This is my port, not an official one.** It matches on everything I have
  thrown at it, and the validation cell above lets you test it on your own
  episodes. If you find a divergence, assume the port is wrong.
- **Pin `kaggle-environments==1.32.6`.** The Kaggle notebook image ships a
  different version whose defaults do not match the competition (see the first
  cell). The exporter also writes the environment's actual configuration into
  each trace and the C++ side reads it back, so config differences are picked up
  rather than assumed — but a different environment *version* can differ in more
  than config, so pin it.
- **Only the fields a strategy reads are modelled.** Some cosmetic observation
  fields are absent.
- **No strategy code here** — environment only.

If you extend or fix it, please share back. A validated fast simulator is more
useful to everyone than a slightly better agent is to any one of us.